# 第76课 · 88 个琴键，其实只有 12 个音——音高、音程与色度轮（chroma wheel）里的十二平均律（12-tone equal temperament, 12-TET）

**学习目标**
- 理解 12 个音高类别（pitch class）和八度等价性（octave equivalence）
- 掌握 MIDI ↔ 频率转换公式
- 理解音程（interval）：两音之间的半音距离及常见音程名称
- 实现色度轮可视化
- 明白为什么 audio AI 用 chroma 而非原始频率

> **开门课**：优先「12 音类 + 八度等价」与 $f = 440\cdot 2^{n/12}$；谐波回链 L07。

← **上一课**　[L75 · ASR 流水线图形化](../7_asr/L75_visual_asr.ipynb)

> 上节课学习了 **ASR 流水线图形化**：波形 → 声谱图 → token → 文字路径可视化。  
> 本课将探讨 **音乐理论速成**。

## 本课剧情：为什么 C3 和 C4 听起来"一样"，却又"不一样"？

钢琴有 88 个键，但音乐只有 **12 种音高类别（pitch class）**，循环出现。中央 C（C4）= 261 Hz，高八度 C（C5）= 523 Hz——频率翻倍，但人耳感知为"同一个音"（只是更高了）。

**这不是巧合，是物理规律**：弦振动频率翻倍，泛音系列完全重叠，大脑把它们归为同一类。这就是"八度等价（octave equivalence）"。

**MIDI 编号系统**：A4（concert A）= MIDI 69 = 440 Hz。每个半音差 1 个 MIDI 编号，每个八度差 12。

**频率公式**：
```
f = 440 × 2^((midi - 69) / 12)
```
- `midi=69` → f = 440 Hz（A4）
- `midi=60` → f = 261.6 Hz（C4，中央C）
- `midi=81` → f = 880 Hz（A5，高一个八度）

**音高类别（chroma）**：把 MIDI 编号"折叠"到 0-11：
```
chroma = midi % 12   （MIDI 60=C4 恰好 60%12=0，天然 C=0,C#=1,...,B=11，无需偏移）
```

色度轮（chroma wheel）是 12 个音高类别排成圆形——相邻是半音关系，相对的是三全音。所有的和弦、调性、和声分析都建立在这个环上。

本节任务：实现两个转换函数，用数学确认你的音乐直觉。

## 音程（Interval）：两音之间的距离

**音程** = 两个音之间相差的**半音（semitone）数**，在 MIDI 空间里就是两个编号之差的绝对值。

| 音程名称 | 半音数 | 示例（从 C 起算） |
|---|:---:|---|
| 小二度 minor 2nd | 1 | C → C# |
| 大二度 major 2nd | 2 | C → D |
| 小三度 minor 3rd | 3 | C → D# |
| 大三度 major 3rd | 4 | C → E |
| 纯四度 perfect 4th | 5 | C → F |
| 增四度 / 三全音 tritone | 6 | C → F# |
| 纯五度 perfect 5th | 7 | C → G |
| 大六度 major 6th | 9 | C → A |
| 纯八度 octave | 12 | C → C（高八度）|

**MIDI 音程公式**：`interval_semitones = |MIDI_高 - MIDI_低|`

例：C4（MIDI 60）到 G4（MIDI 67）= 7 个半音 = 纯五度。

> **思考题**：A3（MIDI 57）到 E4（MIDI 64）相差几个半音？这是什么音程？  
> （答：|64-57|=7 半音 = 纯五度）


In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [2]:
import numpy as np
import matplotlib.pyplot as plt

PITCH_CLASSES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
MIDI_A4, FREQ_A4 = 69, 440.0

## ✏️ 任务 1：MIDI ↔ 频率转换

**签名**：
```python
def midi_to_freq(midi: float) -> float:
    # 返回: 频率 Hz；A4=69→440Hz
```

**公式**：`f = 440 × 2^((midi - 69) / 12)`

| MIDI | 音名 | 期望频率 |
|---|---|---|
| 69 | A4 | 440.00 Hz |
| 60 | C4（中央C） | 261.63 Hz |
| 81 | A5 | 880.00 Hz |
| 57 | A3 | 220.00 Hz |

**验收标准**：
- `abs(midi_to_freq(69) - 440.0) < 0.01`
- `abs(midi_to_freq(81) / midi_to_freq(69) - 2.0) < 1e-9`（高八度 = 频率翻倍）

In [ ]:
def midi_to_freq(midi: float) -> float:
    """MIDI 编号 → 频率 Hz。A4=69=440Hz。"""
    # ✏️ TODO: f(n) = 440 × 2^((n-69)/12)
    raise NotImplementedError("TODO")

def freq_to_midi(freq: float) -> float:
    """频率 Hz → 浮点 MIDI 编号。"""
    # ✏️ TODO: 逆公式：n = 69 + 12 × log2(f / 440)
    raise NotImplementedError("TODO")

# 验证
for name, midi, expected in [("A4",69,440.0),("C4",60,261.63),("A3",57,220.0)]:
    got = midi_to_freq(midi)
    ok = abs(got - expected) < 0.5
    print(f"{name} MIDI{midi}: {got:.2f} Hz  {chr(9989) if ok else chr(10060)}")
    assert ok, f"{name} MIDI{midi}: 期望 {expected:.2f} Hz，实际 {got:.2f} Hz — 检查公式 f(n)=440×2^((n-69)/12)"

assert abs(freq_to_midi(440.0) - 69) < 0.01, "freq_to_midi(440) 应返回 69"
assert abs(midi_to_freq(0) - 8.18) < 0.1, "MIDI 0（最低音）应约为 8.18 Hz"
print("✅ 频率转换验证通过")


## ✏️ 任务 2：chroma_from_freq

**签名**：
```python
def chroma_from_freq(freq: float) -> int:
    # 返回: 0-11（C=0, C#=1, D=2, ..., B=11）
```

**3步实现路线**：

| 步骤 | 操作 | 公式 |
|---|---|---|
| 1 | 频率 → MIDI 编号（反函数） | `midi = 69 + 12 × log2(freq / 440)` |
| 2 | MIDI → 音高类别（折叠八度） | `chroma_raw = round(midi) % 12` |
| 3 | 无需偏移（C=0 天然成立） | MIDI 60（C4）% 12 = 0；自查 A4=69 → 69%12=9=A |

**验收标准**：
- `chroma_from_freq(440.0)` == 9（A4→A=chroma 9）
- `chroma_from_freq(261.63)` == 0（C4→C=chroma 0）
- `chroma_from_freq(880.0)` == 9（A5与A4同chroma）

In [ ]:
def chroma_from_freq(freq: float) -> int:
    """频率 Hz → 音高类别 0-11 (C=0, C#=1, ..., B=11)。"""
    # ✏️ TODO: 用 freq_to_midi 换算后 round() 取整，再取 mod 12
    # ⚠️ 必须用 round() 而非 int()（截断）：
    #   B4=493.88Hz → freq_to_midi ≈ 70.9999
    #   int(70.9999)=70 → A# (错误)，round(70.9999)=71 → B (正确) ✓
    # 参考实现：n = round(freq_to_midi(freq)); return n % 12
    raise NotImplementedError("TODO")

# 验证（含 B4 取整边界用例）
for name, freq, expected_pc in [("C4",261.63,0),("A4",440.0,9),("A5",880.0,9),("B4",493.88,11)]:
    pc = chroma_from_freq(freq)
    print(f"{name} ({freq:.1f}Hz): chroma={pc} ({PITCH_CLASSES[pc]}) {chr(9989) if pc==expected_pc else chr(10060)}")
    assert pc == expected_pc, (
        f"{name} chroma 错误：期望 {expected_pc} ({PITCH_CLASSES[expected_pc]})，"
        f"实际 {pc} — 确认用了 round() 而非 int()"
    )

print("✅ chroma 验证通过（含 B4 取整边界）")


In [ ]:
from pathlib import Path
# 🎨 色度轮可视化（教学图表，aviz 不涵盖此内容）
fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={'projection': 'polar'})
angles = np.linspace(0, 2*np.pi, 12, endpoint=False)
colors = plt.cm.hsv(np.linspace(0, 1, 12, endpoint=False))
ax.bar(angles, np.ones(12), width=2*np.pi/12, color=colors, alpha=0.75, align='edge')
for i, (a, name) in enumerate(zip(angles + np.pi/12, PITCH_CLASSES)):
    ax.text(a, 1.35, name, ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_yticklabels([])
ax.set_xticklabels([])
ax.set_title('色度轮 — 12 音高类别', pad=15)
plt.tight_layout()
plt.savefig(Path('chroma_wheel.png'), dpi=80, bbox_inches='tight')
plt.show()

## 大调音阶与 chroma

C 大调 = C-D-E-F-G-A-B = chroma {0,2,4,5,7,9,11}。

同一首曲子升调后转调（transposition），整个 chroma 向量旋转。
不同调性（tonality）的歌曲，chroma 分布形状不同但"模式"相似。

这就是为什么 chroma 向量能做**调性无关的**音乐相似度比较。

## 本课收束

| 概念 | 要记住 |
|---|---|
| 12-EDO | 八度=12半音，相邻半音频率比 = 2^(1/12) ≈ 1.0595 |
| MIDI | f(n)=440×2^((n-69)/12)，A4=MIDI69=440Hz |
| Chroma | MIDI mod 12，消除八度，保留音高类别 |
| L77 | 把 chroma 做到 STFT 上，得到时变 chromagram，调用 aurora.music |

下一课（L77）用 Aurora music 模块提取色度（chroma）、RMS、ZCR 等特征，把音乐理论转换成可计算的数值向量。

## ✏️ 白板挑战：音高数学手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：MIDI=72 是什么音（C4=60，每个八度12个半音）？频率是多少？  
（72=60+12=C5（高一个八度）；f=440×2^((72-69)/12)=440×2^(3/12)≈523.3 Hz）

**问 2**：440 Hz 的音名是什么？它的 chroma 值是几？  
（A4，chroma=9；A=La=MIDI 69，69%12=9，C=0时A=9）

**问 3**：音程（interval）"纯五度"（perfect fifth）= 7个半音。C4→G4 频率比是多少？  
（2^(7/12)≈1.498≈3/2，这就是纯五度为什么听起来"和谐"的物理原因）

**问 4**：若一首曲子从C大调移调（transpose）到D大调（+2个半音），所有MIDI编号如何变化？  
（每个MIDI编号+2；频率乘以2^(2/12)≈1.122；chroma向右移2位）

**问 5**：色度轮（chroma wheel）上，C和F#（升Fa）相差几个半音？这个音程叫什么？  
（C=0，F#=6，相差6个半音；这叫"三全音"（tritone），是最不和谐的音程，恰好把八度等分为两半）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：MIDI=72频率
try:
    freq_72 = midi_to_freq(72)
    expected_72 = 440 * 2 ** ((72 - 69) / 12)
    assert abs(freq_72 - expected_72) < 0.01, f"midi_to_freq(72)={freq_72:.2f}，期望{expected_72:.2f}"
    print(f"Q1 ✅  midi_to_freq(72=C5)={freq_72:.2f} Hz（≈523.25 Hz）")
except (NotImplementedError, TypeError):
    expected_72 = 440 * 2 ** ((72 - 69) / 12)
    print(f"Q1 ✅  C5(MIDI=72): f=440×2^(3/12)≈{expected_72:.2f} Hz（待实现后验证）")

# 问2：440Hz的chroma
try:
    chroma_a4 = chroma_from_freq(440.0)
    assert chroma_a4 == 9, f"440Hz→chroma应=9(A)，得{chroma_a4}"
    print(f"Q2 ✅  chroma_from_freq(440Hz=A4)={chroma_a4}（A=chroma 9，C=0）")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  440Hz=A4，chroma=9（MIDI 69，69%12=9；C=0时A=9）（待实现后验证）")

# 问3：纯五度频率比
perfect_fifth_ratio = 2 ** (7/12)
just_fifth = 3/2
print(f"Q3 ✅  纯五度（7半音）频率比=2^(7/12)≈{perfect_fifth_ratio:.6f}，Just Intonation=3/2={just_fifth:.6f}，差异={abs(perfect_fifth_ratio-just_fifth)*100:.3f}%")
assert abs(perfect_fifth_ratio - just_fifth) < 0.01, "十二平均律纯五度与纯律误差应<1%"

# 问4：移调验证
try:
    # D大调=C大调+2半音
    c_major = [60, 62, 64, 65, 67, 69, 71, 72]  # MIDI C大调
    d_major_midi = [m + 2 for m in c_major]
    d_major_freq = [midi_to_freq(m) for m in d_major_midi]
    c_major_freq = [midi_to_freq(m) for m in c_major]
    ratios = [df/cf for df, cf in zip(d_major_freq, c_major_freq)]
    expected_ratio = 2 ** (2/12)
    assert all(abs(r - expected_ratio) < 1e-9 for r in ratios), "所有频率比应=2^(2/12)"
    print(f"Q4 ✅  移调+2半音：所有频率×{expected_ratio:.4f}（=2^(2/12)）")
except (NotImplementedError, TypeError):
    print(f"Q4 ✅  移调+2半音：所有MIDI+2，频率×2^(2/12)≈1.122（待实现后验证）")

# 问5：三全音（tritone）
c_chroma, f_sharp_chroma = 0, 6
interval = abs(f_sharp_chroma - c_chroma)
assert interval == 6, f"C到F#应=6个半音，得{interval}"
print(f"Q5 ✅  C(chroma={c_chroma})到F#(chroma={f_sharp_chroma})={interval}个半音=三全音（将八度等分为两半，最不和谐音程）")
print("\n🎉 音高数学白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l76_review = {
    "octave_equivalence":      None,  # 理解八度等价：频率翻倍→同名音，chroma=midi%12（调整后）？True/False
    "midi_freq_formula":       None,  # 记住公式 f=440×2^((midi-69)/12)，能手推A3/C4/C5？True/False
    "midi_to_freq_impl":       None,  # midi_to_freq()实现正确，3个验收标准通过？True/False
    "chroma_from_freq_impl":   None,  # chroma_from_freq()实现正确，A4→9, C4→0, A5→9？True/False
    "whiteboard_passed":       None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l76_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l76_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L76 全部通关！进入 L77：音乐特征工程')

---

→ **下一课**　[L77 · 音乐特征工程](L77_music_features.ipynb)

> 下节课将学习 **音乐特征工程**：chroma、RMS 能量、ZCR，调用 aurora.music 从零实现。